# Full Corpus Build – AIC 2026 ASR Retrieval

This notebook builds the real ASR retrieval corpus from the competition videos.

The workflow is intentionally direct: the notebook imports `production/scripts/full_corpus.py` and calls its Python functions in the current process. Long-running work stays visible in the cell, and Phase 1 transcript checkpoints remain on Drive if Colab disconnects or the run is interrupted.

The notebook does not touch `production_smoke/` and does not activate a new release automatically.

## 1. Setup

Mount Drive, define the project paths, and check the source locations before any corpus work starts.

In [1]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import importlib.metadata as importlib_metadata
import json
import shutil
import sys

PROJECT_ROOT     = Path("/content/drive/MyDrive/aic26/asr_model_comparison")
VIDEO_ROOT       = Path("/content/drive/MyDrive/aic25/AIC_Video")
FPS_MAP_PATH     = Path("/content/drive/MyDrive/aic26/fps_map.json")
FULL_CORPUS_ROOT = PROJECT_ROOT / "production_full_corpus"
PRODUCTION_ROOT  = PROJECT_ROOT / "production"
SCRIPTS_ROOT     = PRODUCTION_ROOT / "scripts"
SCRIPT_PATH      = SCRIPTS_ROOT / "full_corpus.py"
RELEASES_ROOT    = FULL_CORPUS_ROOT / "phase2" / "releases"
REPORTS_ROOT     = FULL_CORPUS_ROOT / "reports"

required_paths = [PROJECT_ROOT, VIDEO_ROOT, FPS_MAP_PATH, PRODUCTION_ROOT / "asr_retrieval", SCRIPT_PATH]
for path in required_paths :
    if (not path.exists()) :
        raise FileNotFoundError(path)

video_resolved    = VIDEO_ROOT.resolve()
artifact_resolved = FULL_CORPUS_ROOT.resolve(strict = False)
if (video_resolved == artifact_resolved or video_resolved in artifact_resolved.parents) :
    raise RuntimeError("FULL_CORPUS_ROOT must not equal or sit inside VIDEO_ROOT")

with FPS_MAP_PATH.open("r", encoding = "utf-8") as file :
    fps_map = json.load(file)

mp4_count = sum(1 for path in VIDEO_ROOT.rglob("*.mp4") if path.is_file())
free_gib  = shutil.disk_usage(PROJECT_ROOT).free / 1024**3

print("PROJECT_ROOT     :", PROJECT_ROOT)
print("VIDEO_ROOT       :", VIDEO_ROOT)
print("FULL_CORPUS_ROOT :", FULL_CORPUS_ROOT)
print("MP4 files        :", f"{mp4_count:,}")
print("fps_map entries  :", f"{len(fps_map):,}")
print("Drive free       :", f"{free_gib:,.2f} GiB")

if (mp4_count != len(fps_map)) :
    print("Count mismatch detected. The inventory step below will show the exact missing and extra IDs.")

Mounted at /content/drive
PROJECT_ROOT     : /content/drive/MyDrive/aic26/asr_model_comparison
VIDEO_ROOT       : /content/drive/MyDrive/aic25/AIC_Video
FULL_CORPUS_ROOT : /content/drive/MyDrive/aic26/asr_model_comparison/production_full_corpus
MP4 files        : 1,478
fps_map entries  : 1,478
Drive free       : 2.33 GiB


### Phase 1 environment

Install the verified offline ASR stack in a fresh Phase 1 runtime. `%pip` runs directly in Colab; there is no Python subprocess wrapper around the notebook workflow.

In [2]:
%pip install Cython packaging "numba-cuda>=0.30.4" "nemo_toolkit[asr]==2.7.3" huggingface_hub soundfile scipy tqdm pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.4/77.4 kB 5.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 8.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.6 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of pyannote-metrics to determine which version is compatible with other requirements. This could take a while.
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.0/6.0 MB 67.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 61.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.2/6.2 MB 97.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.0/9

In [3]:
import shutil
import sys

import nemo

from nemo.utils import logging as nemo_logging
nemo_logging.set_verbosity(nemo_logging.ERROR)

import numba
import numba_cuda
import torch

version_parts = tuple(int(part) for part in torch.__version__.split("+")[0].split(".")[ : 2])

if (version_parts < (2, 7)) :
    raise RuntimeError(f"Production Parakeet requires PyTorch >= 2.7; found {torch.__version__}")

if (not torch.cuda.is_available()) :
    raise RuntimeError("CUDA is not available. Select a GPU runtime before Phase 1.")

for executable in ["ffmpeg", "ffprobe"] :
    if (shutil.which(executable) is None) :
        raise RuntimeError(f"Required executable is missing: {executable}")

print("Python     :", sys.version.split()[0])
print("PyTorch    :", torch.__version__)
print("CUDA       :", torch.version.cuda)
print("GPU        :", torch.cuda.get_device_name(0))
print("NeMo       :", nemo.__version__)
print("Numba      :", numba.__version__)
print("Numba CUDA :", numba_cuda.__version__)
print("ffmpeg     :", shutil.which("ffmpeg"))
print("ffprobe    :", shutil.which("ffprobe"))

Python     : 3.12.13
PyTorch    : 2.11.0+cu128
CUDA       : 12.8
GPU        : Tesla T4
NeMo       : 2.7.3
Numba      : 0.60.0
Numba CUDA : 0.30.4
ffmpeg     : /usr/bin/ffmpeg
ffprobe    : /usr/bin/ffprobe


### Import the corpus helpers

The notebook calls these functions directly. This cell is also the only helper-import cell you need to rerun after the Phase 2 runtime restart.

In [4]:
import pandas as pd

if (str(SCRIPTS_ROOT) not in sys.path) :
    sys.path.insert(0, str(SCRIPTS_ROOT))

from full_corpus import (
    activate_corpus_release,
    audit_phase1,
    benchmark_release,
    build_corpus_release,
    load_inventory,
    make_corpus_build_config,
    make_offline_config,
    phase1_acceptance,
    prepare_inventory,
    read_json,
    scan_phase1,
    select_pilot_ids,
    transcribe_corpus,
    validate_corpus_release,
)
from asr_retrieval.config import parakeet_identity, window_policy_identity
from asr_retrieval.postprocess import POSTPROCESS_VERSION

offline_config = make_offline_config(FULL_CORPUS_ROOT)
corpus_config  = make_corpus_build_config(FULL_CORPUS_ROOT)

print("Parakeet model        :", offline_config.parakeet.model_name)
print("Parakeet revision     :", offline_config.parakeet.revision)
print("Parakeet identity     :", parakeet_identity(offline_config.parakeet))
print("Window seconds        :", offline_config.audio.window_seconds)
print("Stride seconds        :", offline_config.audio.stride_seconds)
print("Window policy identity:", window_policy_identity(offline_config.audio))
print("Postprocess version   :", POSTPROCESS_VERSION)
print("BM25 k1 / b           :", corpus_config.bm25.k1, "/", corpus_config.bm25.b)
print("E5 model              :", corpus_config.e5.model_name)
print("E5 revision           :", corpus_config.e5.revision)

Parakeet model        : nvidia/parakeet-ctc-0.6b-Vietnamese
Parakeet revision     : b0493142b49458810324e3db8be9e8e07b4ebc17
Parakeet identity     : {'model_name': 'nvidia/parakeet-ctc-0.6b-Vietnamese', 'model_filename': 'parakeet-ctc-0.6b-vi.nemo', 'revision': 'b0493142b49458810324e3db8be9e8e07b4ebc17', 'timestamps': True, 'decoder': 'greedy_ctc', 'dtype': 'float32', 'batch_size': 1}
Window seconds        : 60.0
Stride seconds        : 45.0
Window policy identity: 2e9bc23da7904c796817114f3cfc438b1d87dafcc08a286337581e66d7ffb679
Postprocess version   : 1.1
BM25 k1 / b           : 1.5 / 0.75
E5 model              : intfloat/multilingual-e5-large-instruct
E5 revision           : 274baa43b0e13e37fafa6428dbc7938e62e5c439


## 2. Inventory and pilot

The inventory is the authoritative list of source videos. The first run freezes it; later runs verify that the MP4 paths, sizes, modification times, and FPS values still match.

### Freeze or verify the inventory

In [ ]:
inventory_rows, inventory_summary = prepare_inventory(
    video_root    = VIDEO_ROOT,
    fps_map_path  = FPS_MAP_PATH,
    artifact_root = FULL_CORPUS_ROOT,
)

inventory_df = pd.DataFrame(inventory_rows)
inventory_df["prefix"] = inventory_df["video_id"].str.split("_").str[0]

print("Inventory identity:", inventory_summary["inventory_identity"])
print("Videos            :", f"{len(inventory_rows):,}")
print("Source size       :", f"{inventory_summary['total_size_bytes'] / 1024**3:,.2f} GiB")
display(inventory_df.head(10))
display(inventory_df.groupby("prefix").size().rename("videos").to_frame())
display((inventory_df["size_bytes"] / 1024**2).describe().rename("size_MiB").to_frame())

Inventory identity: 466408aa4d89e7e0412def939a5281081c17dfa1c151f8bf34e4538e7e1b23b4
Videos            : 1,478
Source size       : 228.31 GiB


,video_id,source_path,relative_source_path,size_bytes,mtime_ns,expected_from_fps_map,fps,prefix
0,K01_V001,/content/drive/MyDrive/aic25/AIC_Video/K01/K01...,K01/K01_V001.mp4,425366086,1757133956000000000,True,25.0,K01
1,K01_V002,/content/drive/MyDrive/aic25/AIC_Video/K01/K01...,K01/K01_V002.mp4,482719300,1757133962000000000,True,30.0,K01
2,K01_V003,/content/drive/MyDrive/aic25/AIC_Video/K01/K01...,K01/K01_V003.mp4,272767047,1757133969000000000,True,30.0,K01
3,K01_V004,/content/drive/MyDrive/aic25/AIC_Video/K01/K01...,K01/K01_V004.mp4,240594463,1757133976000000000,True,25.0,K01
4,K01_V005,/content/drive/MyDrive/aic25/AIC_Video/K01/K01...,K01/K01_V005.mp4,440495474,1757133981000000000,True,25.0,K01
5,K01_V006,/content/drive/MyDrive/aic25/AIC_Video/K01/K01...,K01/K01_V006.mp4,386351530,1757133989000000000,True,30.0,K01
6,K01_V007,/content/drive/MyDrive/aic25/AIC_Video/K01/K01...,K01/K01_V007.mp4,438690481,1757133997000000000,True,30.0,K01
7,K01_V008,/content/drive/MyDrive/aic25/AIC_Video/K01/K01...,K01/K01_V008.mp4,460110435,1757134006000000000,True,30.0,K01
8,K01_V009,/content/drive/MyDrive/aic25/AIC_Video/K01/K01...,K01/K01_V009.mp4,272572902,1757134015000000000,True,30.0,K01
9,K01_V010,/content/drive/MyDrive/aic25/AIC_Video/K01/K01...,K01/K01_V010.mp4,532551829,1757134020000000000,True,30.0,K01


,videos
prefix,
K01,31
K02,31
K03,29
K04,30
K05,31
K06,31
K07,31
K08,30
K09,28


,size_MiB
count,1478.000000
mean,158.180712
std,106.350069
min,9.037562
25%,66.123909
50%,135.144695
75%,233.188612
max,589.241631


### Pilot

The pilot selects eight deterministic videos spread across corpus prefixes. It writes into the real Phase 1 workspace, so successful pilot artifacts are reused by the full run.

In [ ]:
pilot_ids = select_pilot_ids(inventory_rows, count = 8)
print("Pilot IDs:", pilot_ids)

pilot_report = transcribe_corpus(
    artifact_root = FULL_CORPUS_ROOT,
    video_ids     = pilot_ids,
)

pilot_status_rows, _ = scan_phase1(FULL_CORPUS_ROOT)
pilot_status_df = pd.DataFrame(pilot_status_rows)
pilot_status_df = pilot_status_df[pilot_status_df["video_id"].isin(pilot_ids)].reset_index(drop = True)
display(pilot_status_df)

if (pilot_report["failures"]) :
    raise RuntimeError(f"Pilot source failures: {pilot_report['failures']}")
if (not pilot_status_df["complete"].all()) :
    raise RuntimeError("One or more pilot artifacts are incomplete")
if (pilot_status_df["eligible_windows"].sum() <= 0) :
    raise RuntimeError("The pilot produced no retrieval-eligible transcript windows")
print("Pilot: PASS")

Pilot IDs: ['K01_V016', 'K05_V016', 'K09_V015', 'K13_V016', 'K18_V015', 'L22_V016', 'L26_V250', 'L30_V049']
Corpus complete : 740 / 1,478
Selected        : 8
Skipped         : 8
To process      : 0
Nothing to transcribe. Parakeet was not loaded.


,video_id,artifact_path,artifact_exists,complete,window_count,ok_nonempty_windows,ok_empty_windows,failed_windows,eligible_windows,classification,invalid_reason
0,K01_V016,/content/drive/MyDrive/aic26/asr_model_compari...,True,True,24,24,0,0,24,complete_clean,None
1,K05_V016,/content/drive/MyDrive/aic26/asr_model_compari...,True,True,18,18,0,0,18,complete_clean,None
2,K09_V015,/content/drive/MyDrive/aic26/asr_model_compari...,True,True,26,26,0,0,26,complete_clean,None
3,K13_V016,/content/drive/MyDrive/aic26/asr_model_compari...,True,True,23,23,0,0,23,complete_clean,None
4,K18_V015,/content/drive/MyDrive/aic26/asr_model_compari...,True,True,24,24,0,0,24,complete_clean,None
5,L22_V016,/content/drive/MyDrive/aic26/asr_model_compari...,True,True,32,32,0,0,32,complete_clean,None
6,L26_V250,/content/drive/MyDrive/aic26/asr_model_compari...,True,True,7,7,0,0,7,complete_clean,None
7,L30_V049,/content/drive/MyDrive/aic26/asr_model_compari...,True,True,4,4,0,0,4,complete_clean,None


Pilot: PASS


In [ ]:
# Rerun the same pilot. Complete videos should be skipped before Parakeet is loaded.
resume_report = transcribe_corpus(
    artifact_root = FULL_CORPUS_ROOT,
    video_ids     = pilot_ids,
)

if (resume_report["work_video_count"] != 0) :
    raise RuntimeError("Pilot resume check failed: completed videos were reopened")
print("Pilot resume check: PASS")

for video_id in pilot_ids :
    transcript_path = FULL_CORPUS_ROOT / "phase1" / "transcripts" / f"{video_id}.json"
    payload         = read_json(transcript_path)
    eligible        = [window for window in payload["windows"] if window.get("eligible")]
    example         = eligible[0]["retrieval_text"] if eligible else "<no retrieval-eligible window>"
    print(f"\n{video_id} | windows={len(payload['windows'])} | eligible={len(eligible)}")
    print(example[ : 400])

Corpus complete : 740 / 1,478
Selected        : 8
Skipped         : 8
To process      : 0
Nothing to transcribe. Parakeet was not loaded.
Pilot resume check: PASS

K01_V016 | windows=24 | eligible=24
h kính chào quý vị đây là chương trình sáu mươi giây chiều của đài truyền hình thành phố hồ chí minh h hôm nay mười sáu tháng mười chương trình sẽ có những nội dung nổi bật sau đây những khu vực tại thành phố hồ chí minh có nguy cơ ngập do triều cường cuối tuần này nỗ lực nhiệt sâu đầu đen gây hại vườ dừa tại tỉnh tiền giang bầu cử mỹ nóng lên từ ngày cử tri một màn chiếm địa đi bỏ phiếu sớm sáng 

K05_V016 | windows=18 | eligible=18
xin kính chào quý vị xin mời quý vị theo dõi chương trình sáu mươi giây sáng đầu tuần của đài truyền hình thành phố hồ chí minh chương trình sáng nay ngày mười sáu tháng mười hai năm hai ngàn không trăm hai mươi bốn sẽ có những nội dung chính sau đây đã đưa được hơn ba trăm năm mươi người bị mắc kẹt do sự cố sạt lở đèo khánh lê về nơi an toàn khá tham quan trả

## 3. Phase 1 transcription

This is the long-running ASR step. It runs directly in the notebook process with visible `tqdm` progress.

Completed videos are skipped. Partial videos are handed back to the existing production `transcribe_video()` logic, which validates the source and resumes from its saved window checkpoints. The cell can be interrupted or lost and rerun later.

In [ ]:
# Long-running cell. Safe to rerun after a Colab disconnect or manual interruption.
transcription_report = transcribe_corpus(
    artifact_root = FULL_CORPUS_ROOT,
)

Corpus complete : 1,170 / 1,478
Selected        : 1,478
Skipped         : 1,170
To process      : 308
Loading Parakeet...
Loaded: nvidia/parakeet-ctc-0.6b-Vietnamese @ b0493142b49458810324e3db8be9e8e07b4ebc17
[1,171/1,478] K19_V003 | FAILED | RuntimeError: Video is already owned by another transcription worker: K19_V003
[1,172/1,478] L25_V007 | FAILED | RuntimeError: Video is already owned by another transcription worker: L25_V007
[1,173/1,478] L26_V351 | DONE | windows=7 | eligible=7 | 7.6s
[1,174/1,478] L26_V352 | DONE | windows=7 | eligible=7 | 10.6s
[1,175/1,478] L26_V353 | DONE | windows=7 | eligible=7 | 11.1s
[1,176/1,478] L26_V354 | DONE | windows=7 | eligible=7 | 11.3s
[1,177/1,478] L26_V355 | DONE | windows=7 | eligible=7 | 9.7s
[1,178/1,478] L26_V356 | DONE | windows=7 | eligible=7 | 10.6s
[1,179/1,478] L26_V357 | DONE | windows=7 | eligible=7 | 10.7s
[1,180/1,478] L26_V358 | DONE | windows=7 | eligible=7 | 10.2s
[1,181/1,478] L26_V359 | DONE | windows=7 | eligible=7 | 10.9s


In [ ]:
# Read-only safety check after an interruption or Drive quota error.
# This does not load Parakeet and does not write anything to Drive.

status_rows, status_summary = scan_phase1(FULL_CORPUS_ROOT)
status_df                   = pd.DataFrame(status_rows)

transcript_root = FULL_CORPUS_ROOT / "phase1" / "transcripts"
canonical_root  = FULL_CORPUS_ROOT / "phase1" / "canonical_audio"
temp_audio_root = FULL_CORPUS_ROOT / "phase1" / "window_audio_tmp"

invalid_df = status_df[status_df["classification"] == "invalid_artifact"]
partial_df = status_df[status_df["classification"] == "partial"]
zero_byte  = [path for path in transcript_root.glob("*.json") if path.stat().st_size == 0]

print("Phase 1 saved state")
print("-------------------")
print(f"Complete videos : {status_summary['complete_videos']:,} / {status_summary['total_expected_videos']:,}")
print(f"Partial videos  : {status_summary['partial_videos']:,}")
print(f"Missing videos  : {status_summary['missing_videos']:,}")
print(f"Invalid files   : {status_summary['invalid_artifacts']:,}")
print(f"Failed windows  : {status_summary['failed_windows']:,}")
print(f"Physical windows: {status_summary['physical_windows']:,}")
print(f"Eligible windows: {status_summary['eligible_windows']:,}")
print(f"Extra JSON files: {len(status_summary['extra_transcript_artifact_ids']):,}")
print(f"Zero-byte JSONs : {len(zero_byte):,}")

print("\nInterrupted area")
focus_ids = ["L26_V348", "L26_V349", "L26_V350", "L26_V351"]
focus_columns = [
    "video_id", "classification", "complete", "window_count",
    "eligible_windows", "failed_windows", "invalid_reason",
]
display(status_df[status_df["video_id"].isin(focus_ids)][focus_columns].reset_index(drop = True))

print("Phase 1 storage currently visible on Drive")
for label, path in [
    ("transcripts", transcript_root),
    ("canonical_audio", canonical_root),
    ("window_audio_tmp", temp_audio_root),
] :
    size_bytes = sum(item.stat().st_size for item in path.rglob("*") if item.is_file()) if path.exists() else 0
    file_count = sum(1 for item in path.rglob("*") if item.is_file()) if path.exists() else 0
    print(f"{label:18s}: {file_count:,} files | {size_bytes / 1024**3:,.3f} GiB")

storage_safe = (
    status_summary["complete_videos"] >= 1_173
    and status_summary["invalid_artifacts"] == 0
    and not status_summary["extra_transcript_artifact_ids"]
    and not zero_byte
)

print()
print("Saved checkpoint check:", "PASS" if storage_safe else "REVIEW REQUIRED")

if (not invalid_df.empty) :
    print("\nInvalid transcript artifacts:")
    display(invalid_df[focus_columns].head(50))

if (not partial_df.empty) :
    print("\nValid partial checkpoints:")
    display(partial_df[focus_columns].head(50))

Phase 1 saved state
-------------------
Complete videos : 1,476 / 1,478
Partial videos  : 1
Missing videos  : 1
Invalid files   : 0
Failed windows  : 0
Physical windows: 26,128
Eligible windows: 26,082
Extra JSON files: 0
Zero-byte JSONs : 0

Interrupted area


,video_id,classification,complete,window_count,eligible_windows,failed_windows,invalid_reason
0,L26_V348,complete_clean,True,7,7,0,None
1,L26_V349,complete_clean,True,7,7,0,None
2,L26_V350,complete_clean,True,7,7,0,None
3,L26_V351,complete_clean,True,7,7,0,None


Phase 1 storage currently visible on Drive
transcripts       : 1,477 files | 0.194 GiB
canonical_audio   : 2 files | 0.082 GiB
window_audio_tmp  : 1 files | 0.002 GiB

Saved checkpoint check: PASS

Valid partial checkpoints:


,video_id,classification,complete,window_count,eligible_windows,failed_windows,invalid_reason
739,L25_V007,partial,False,53,53,0,None


In [ ]:
# Fast reconnect check: reads saved transcript JSON only. No Parakeet model load and no source hashing.
status_rows, status_summary = scan_phase1(FULL_CORPUS_ROOT)

print("Complete        :", f"{status_summary['complete_videos']:,} / {status_summary['total_expected_videos']:,}")
print("Partial         :", f"{status_summary['partial_videos']:,}")
print("Missing         :", f"{status_summary['missing_videos']:,}")
print("Invalid         :", f"{status_summary['invalid_artifacts']:,}")
print("Failed windows  :", f"{status_summary['failed_windows']:,}")
print("Empty OK windows:", f"{status_summary['empty_ok_windows']:,}")
print("Eligible windows:", f"{status_summary['eligible_windows']:,}")
display(pd.DataFrame(status_summary["classification_counts"].items(), columns = ["classification", "videos"]))

Complete        : 1,476 / 1,478
Partial         : 1
Missing         : 1
Invalid         : 0
Failed windows  : 0
Empty OK windows: 15
Eligible windows: 26,082


,classification,videos
0,complete_clean,1466
1,complete_with_empty_windows,10
2,missing,1
3,partial,1


In [ ]:
# Full Phase 1 verification against the actual source videos.
# Read-only: no ASR model load and no transcript files are modified.

from asr_retrieval.audio import build_physical_windows, probe_video, select_audio_stream
from full_corpus import inspect_source_inventory
from tqdm.auto import tqdm

inventory_rows, inventory_summary = load_inventory(FULL_CORPUS_ROOT)
status_rows, status_summary       = scan_phase1(FULL_CORPUS_ROOT)
offline_config                    = make_offline_config(FULL_CORPUS_ROOT)

# Recheck the actual 1,478 MP4 files, sizes, mtimes and FPS values.
current_inventory_rows, current_inventory_summary = inspect_source_inventory(VIDEO_ROOT, FPS_MAP_PATH)

if (not current_inventory_summary["passed"]) :
    raise RuntimeError("Current source inventory failed validation.")

if (current_inventory_summary["inventory_identity"] != inventory_summary["inventory_identity"]) :
    raise RuntimeError("Current source inventory no longer matches the frozen inventory.")

status_by_id    = {row["video_id"] : row for row in status_rows}
validation_rows = []
probe_errors    = []

for inventory_row in tqdm(inventory_rows, desc = "Validating videos", unit = "video", dynamic_ncols = True) :
    video_id        = inventory_row["video_id"]
    source_path     = Path(inventory_row["source_path"])
    status          = status_by_id[video_id]
    transcript_path = FULL_CORPUS_ROOT / "phase1" / "transcripts" / f"{video_id}.json"

    video_duration_s = None
    audio_duration_s = None
    probe_error      = None

    try :
        probe        = probe_video(source_path)
        audio_stream = select_audio_stream(probe)

        raw_video_duration = probe.get("format", {}).get("duration")
        raw_audio_duration = audio_stream.get("duration")

        if (raw_video_duration not in {None, "N/A"}) :
            video_duration_s = float(raw_video_duration)

        if (raw_audio_duration not in {None, "N/A"}) :
            audio_duration_s = float(raw_audio_duration)
    except Exception as error :
        probe_error = f"{type(error).__name__}: {error}"
        probe_errors.append({"video_id" : video_id, "error" : probe_error})

    saved_source_duration_s = None
    canonical_duration_s    = None
    observed_windows        = int(status["window_count"])
    expected_windows_exact  = None

    if (status["artifact_exists"] and status["classification"] != "invalid_artifact") :
        payload         = read_json(transcript_path)
        video_metadata  = payload["video"]
        canonical_audio = payload["canonical_audio"]

        saved_source_duration_s = video_metadata.get("source_duration_s")
        sample_rate             = int(canonical_audio["sample_rate"])
        sample_count            = int(canonical_audio["sample_count"])
        canonical_duration_s    = sample_count / sample_rate
        expected_windows_exact  = len(build_physical_windows(video_id, sample_count, offline_config.audio))

    duration_reference_s = audio_duration_s if audio_duration_s is not None else video_duration_s
    expected_windows_probe = None

    if (duration_reference_s is not None and duration_reference_s > 0.0) :
        estimated_samples      = max(1, round(duration_reference_s * offline_config.audio.sample_rate))
        expected_windows_probe = len(build_physical_windows(video_id, estimated_samples, offline_config.audio))

    exact_window_match = (
        observed_windows == expected_windows_exact
        if status["complete"] and expected_windows_exact is not None
        else observed_windows <= expected_windows_exact
        if expected_windows_exact is not None
        else None
    )

    saved_duration_match = None
    if (saved_source_duration_s is not None and video_duration_s is not None) :
        saved_duration_match = abs(float(saved_source_duration_s) - video_duration_s) <= 0.1

    duration_gap_s   = None
    duration_close   = None
    duration_tolerance_s = None

    if (canonical_duration_s is not None and duration_reference_s is not None) :
        duration_gap_s       = abs(canonical_duration_s - duration_reference_s)
        duration_tolerance_s = max(2.0, 0.01 * duration_reference_s)
        duration_close       = duration_gap_s <= duration_tolerance_s

    validation_rows.append({
        "video_id"                  : video_id,
        "classification"            : status["classification"],
        "complete"                  : status["complete"],
        "video_duration_s"          : video_duration_s,
        "audio_duration_s"          : audio_duration_s,
        "canonical_duration_s"      : canonical_duration_s,
        "observed_windows"          : observed_windows,
        "expected_windows_exact"    : expected_windows_exact,
        "expected_windows_probe"    : expected_windows_probe,
        "exact_window_match"        : exact_window_match,
        "saved_duration_match"      : saved_duration_match,
        "duration_gap_s"            : duration_gap_s,
        "duration_close"            : duration_close,
        "eligible_windows"          : status["eligible_windows"],
        "failed_windows"            : status["failed_windows"],
        "empty_ok_windows"          : status["ok_empty_windows"],
        "probe_error"               : probe_error,
        "invalid_reason"            : status["invalid_reason"],
    })

validation_df = pd.DataFrame(validation_rows)

invalid_ids  = validation_df.loc[validation_df["classification"] == "invalid_artifact", "video_id"].tolist()
recovery_ids = validation_df.loc[~validation_df["complete"] & (validation_df["classification"] != "invalid_artifact"), "video_id"].tolist()
failed_ids   = validation_df.loc[validation_df["failed_windows"] > 0, "video_id"].tolist()

exact_window_errors = validation_df[
    validation_df["complete"]
    & validation_df["exact_window_match"].eq(False)
]

duration_review_df = validation_df[
    validation_df["complete"]
    & validation_df["duration_close"].eq(False)
]

print("Full Phase 1 verification")
print("-------------------------")
print(f"Inventory match      : PASS")
print(f"Expected videos      : {len(inventory_rows):,}")
print(f"Complete             : {status_summary['complete_videos']:,}")
print(f"Partial              : {status_summary['partial_videos']:,}")
print(f"Missing              : {status_summary['missing_videos']:,}")
print(f"Invalid artifacts    : {len(invalid_ids):,}")
print(f"Exact window errors  : {len(exact_window_errors):,}")
print(f"FFprobe errors       : {len(probe_errors):,}")
print(f"Duration reviews     : {len(duration_review_df):,}")
print(f"Failed ASR videos    : {len(failed_ids):,}")
print(f"Need transcription   : {len(recovery_ids):,}")

print("\nWindow-count distribution")
window_distribution = (
    validation_df[validation_df["complete"]]
    .groupby("observed_windows")
    .agg(
        videos                = ("video_id", "count"),
        min_video_duration_s  = ("video_duration_s", "min"),
        max_video_duration_s  = ("video_duration_s", "max"),
        min_audio_duration_s  = ("canonical_duration_s", "min"),
        max_audio_duration_s  = ("canonical_duration_s", "max"),
    )
    .reset_index()
    .sort_values("observed_windows")
)
display(window_distribution)

print("\nVideos with 7 or fewer windows")
low_window_df = validation_df[
    validation_df["complete"] & (validation_df["observed_windows"] <= 7)
][[
    "video_id",
    "video_duration_s",
    "audio_duration_s",
    "canonical_duration_s",
    "observed_windows",
    "expected_windows_exact",
    "expected_windows_probe",
    "exact_window_match",
]]
display(low_window_df.sort_values(["observed_windows", "video_id"]).reset_index(drop = True))

if (recovery_ids) :
    print("\nVideos that still need transcription:")
    display(validation_df[validation_df["video_id"].isin(recovery_ids)][[
        "video_id", "classification", "observed_windows", "expected_windows_exact", "probe_error", "invalid_reason"
    ]].reset_index(drop = True))

if (invalid_ids) :
    print("\nINVALID artifacts – do not automatically overwrite these:")
    display(validation_df[validation_df["video_id"].isin(invalid_ids)][[
        "video_id", "classification", "invalid_reason"
    ]].reset_index(drop = True))

if (not exact_window_errors.empty) :
    print("\nExact window-count mismatches:")
    display(exact_window_errors)

if (duration_review_df.shape[0] > 0) :
    print("\nSource/audio duration differences to review:")
    display(duration_review_df[[
        "video_id", "video_duration_s", "audio_duration_s", "canonical_duration_s",
        "observed_windows", "expected_windows_exact", "duration_gap_s"
    ]].sort_values("duration_gap_s", ascending = False).head(50))

validation_passed = (
    current_inventory_summary["inventory_identity"] == inventory_summary["inventory_identity"]
    and not invalid_ids
    and exact_window_errors.empty
    and not probe_errors
)

print()
print("Structural + source verification:", "PASS" if validation_passed else "REVIEW REQUIRED")

Validating videos:   0%|          | 0/1478 [00:00<?, ?video/s]

Full Phase 1 verification
-------------------------
Inventory match      : PASS
Expected videos      : 1,478
Complete             : 1,476
Partial              : 1
Missing              : 1
Invalid artifacts    : 0
Exact window errors  : 0
FFprobe errors       : 0
Duration reviews     : 0
Failed ASR videos    : 0
Need transcription   : 2

Window-count distribution


,observed_windows,videos,min_video_duration_s,max_video_duration_s,min_audio_duration_s,max_audio_duration_s
0,1,4,32.555,37.779,32.554375,37.778875
1,2,12,65.179,97.222,65.178438,97.221937
2,3,16,107.718,147.633,107.717375,147.632500
3,4,16,160.172,194.978,160.171250,194.978000
4,5,17,197.881,235.335,197.880437,235.334250
5,6,24,240.141,282.425,240.140813,282.424312
6,7,472,287.580,329.910,287.579125,329.909125
7,8,49,330.026,371.497,330.025250,371.496062
8,9,10,375.142,417.054,375.141625,417.053625
9,10,2,421.698,435.537,421.697625,435.536688



Videos with 7 or fewer windows


,video_id,video_duration_s,audio_duration_s,canonical_duration_s,observed_windows,expected_windows_exact,expected_windows_probe,exact_window_match
0,L24_V040,37.779,37.778866,37.778875,1,1.0,1,True
1,L24_V041,35.852,35.851610,35.851625,1,1.0,1,True
2,L24_V044,35.341,35.340771,35.340812,1,1.0,1,True
3,L24_V045,32.555,32.554376,32.554375,1,1.0,1,True
4,L24_V012,97.222,97.221950,97.221937,2,2.0,2,True
...,...,...,...,...,...,...,...,...
556,L30_V052,292.758,292.757188,292.757187,7,7.0,7,True
557,L30_V053,304.368,304.367166,304.367188,7,7.0,7,True
558,L30_V056,300.351,300.350113,300.350125,7,7.0,7,True
559,L30_V083,297.425,297.424399,297.424438,7,7.0,7,True



Videos that still need transcription:


,video_id,classification,observed_windows,expected_windows_exact,probe_error,invalid_reason
0,K19_V003,missing,0,NaN,None,None
1,L25_V007,partial,53,61.0,None,None



Structural + source verification: PASS


In [ ]:
# Recover only unfinished videos.
# Clear stale locks only when no other transcription worker is running.

if (invalid_ids) :
    raise RuntimeError(f"Invalid transcript artifacts require manual review first: {invalid_ids}")

if (probe_errors) :
    raise RuntimeError(f"Source probe errors require review first: {probe_errors[ : 10]}")

LOCK_ROOT = FULL_CORPUS_ROOT / "phase1" / "locks"

recovery_lock_paths = [
    LOCK_ROOT / f"{video_id}.lock"
    for video_id in recovery_ids
    if (LOCK_ROOT / f"{video_id}.lock").exists()
]

print("Videos needing transcription:", recovery_ids)
print("Locks blocking recovery      :", len(recovery_lock_paths))

for lock_path in recovery_lock_paths :
    print(f"{lock_path.stem}: {lock_path.read_text(encoding = 'utf-8', errors = 'replace')}")

CLEAR_STALE_LOCKS = True  # Only when no other Colab/account is transcribing.

if (recovery_lock_paths and not CLEAR_STALE_LOCKS) :
    raise RuntimeError("Stale locks were found. Confirm no transcription worker is active before clearing them.")

for lock_path in recovery_lock_paths :
    lock_path.unlink()
    print("Removed stale lock:", lock_path.name)

if (recovery_ids) :
    recovery_report = transcribe_corpus(
        artifact_root = FULL_CORPUS_ROOT,
        video_ids     = recovery_ids,
    )
else :
    print("No missing or partial videos need transcription.")

Videos needing transcription: ['K19_V003', 'L25_V007']
Locks blocking recovery      : 2
K19_V003: {"pid": 16578, "host": "54052b8ea5e2", "created_at_utc": "2026-08-18T07:51:54.200657+00:00"}
L25_V007: {"pid": 3124, "host": "f333ed127747", "created_at_utc": "2026-08-18T09:11:15.380051+00:00"}
Removed stale lock: K19_V003.lock
Removed stale lock: L25_V007.lock
Corpus complete : 1,476 / 1,478
Selected        : 2
Skipped         : 0
To process      : 2
Loading Parakeet...
Loaded: nvidia/parakeet-ctc-0.6b-Vietnamese @ b0493142b49458810324e3db8be9e8e07b4ebc17
[    1/2] K19_V003 | DONE | windows=27 | eligible=27 | 24.1s
[    2/2] L25_V007 | DONE | windows=61 | eligible=61 | 14.3s

Completed this run : 2
Corpus complete    : 1,478 / 1,478
Source failures    : 0


In [ ]:
status_rows, status_summary = scan_phase1(FULL_CORPUS_ROOT)
status_df                   = pd.DataFrame(status_rows)

print("Complete       :", f"{status_summary['complete_videos']:,} / {status_summary['total_expected_videos']:,}")
print("Partial        :", f"{status_summary['partial_videos']:,}")
print("Missing        :", f"{status_summary['missing_videos']:,}")
print("Invalid        :", f"{status_summary['invalid_artifacts']:,}")
print("Failed windows :", f"{status_summary['failed_windows']:,}")
print("Empty OK       :", f"{status_summary['empty_ok_windows']:,}")
print("Eligible       :", f"{status_summary['eligible_windows']:,}")

remaining_df = status_df[
    (~status_df["complete"])
    | (status_df["classification"] == "invalid_artifact")
    | (status_df["failed_windows"] > 0)
]

if (not remaining_df.empty) :
    display(remaining_df.reset_index(drop = True))
    raise RuntimeError("Phase 1 is not fully clean yet.")

print("Phase 1 corpus verification: PASS")

Complete       : 1,478 / 1,478
Partial        : 0
Missing        : 0
Invalid        : 0
Failed windows : 0
Empty OK       : 15
Eligible       : 26,117
Phase 1 corpus verification: PASS


### Audit and retries

Audit the complete Phase 1 workspace, retry failed ASR windows, and only retry empty successful windows when you have explicitly reviewed those videos. Silence and non-speech can legitimately produce empty transcripts.

In [ ]:
audit_rows, failed_window_rows, phase1_summary = audit_phase1(FULL_CORPUS_ROOT)

print(json.dumps(phase1_summary, ensure_ascii = False, indent = 2))
if (failed_window_rows) :
    display(pd.DataFrame(failed_window_rows).head(100))
else :
    print("No failed ASR windows found.")

{
  "inventory_identity": "466408aa4d89e7e0412def939a5281081c17dfa1c151f8bf34e4538e7e1b23b4",
  "total_expected_videos": 1478,
  "complete_videos": 1478,
  "partial_videos": 0,
  "missing_videos": 0,
  "invalid_artifacts": 0,
  "failed_windows": 0,
  "empty_ok_windows": 15,
  "eligible_windows": 26117,
  "physical_windows": 26163,
  "extra_transcript_artifact_ids": [],
  "classification_counts": {
    "complete_clean": 1468,
    "complete_with_empty_windows": 10
  },
  "created_at_utc": "2026-08-18T20:07:57.276803+00:00",
  "passed": true
}
No failed ASR windows found.


In [ ]:
failed_video_ids = sorted({row["video_id"] for row in failed_window_rows})

if (failed_video_ids) :
    print(f"Retrying failed windows in {len(failed_video_ids):,} videos...")
    retry_report = transcribe_corpus(
        artifact_root = FULL_CORPUS_ROOT,
        video_ids     = failed_video_ids,
        retry_failed  = True,
    )
    audit_rows, failed_window_rows, phase1_summary = audit_phase1(FULL_CORPUS_ROOT)
else :
    print("No failed ASR windows remain.")

print("Residual failed windows:", phase1_summary["failed_windows"])

No failed ASR windows remain.
Residual failed windows: 0


In [ ]:
# Add IDs here only after reviewing empty-window cases that should contain speech.
EMPTY_RETRY_VIDEO_IDS : list[str] = []

if (EMPTY_RETRY_VIDEO_IDS) :
    empty_retry_report = transcribe_corpus(
        artifact_root = FULL_CORPUS_ROOT,
        video_ids     = EMPTY_RETRY_VIDEO_IDS,
        retry_empty   = True,
    )
    audit_rows, failed_window_rows, phase1_summary = audit_phase1(FULL_CORPUS_ROOT)
else :
    print("No selective empty-window retry requested.")

No selective empty-window retry requested.


In [ ]:
ALLOW_RESIDUAL_FAILURES = False

acceptance = phase1_acceptance(
    artifact_root           = FULL_CORPUS_ROOT,
    allow_residual_failures = ALLOW_RESIDUAL_FAILURES,
)
print(json.dumps(acceptance, ensure_ascii = False, indent = 2))

{
  "schema_version": "1.0",
  "accepted_at_utc": "2026-08-18T20:08:15.284251+00:00",
  "inventory_identity": "466408aa4d89e7e0412def939a5281081c17dfa1c151f8bf34e4538e7e1b23b4",
  "inventory_verified_at_utc": "2026-08-18T20:08:01.912774+00:00",
  "total_expected_videos": 1478,
  "complete_videos": 1478,
  "physical_window_count": 26163,
  "eligible_window_count": 26117,
  "residual_failed_window_count": 0,
  "residual_failed_window_ids": [],
  "residual_failures_explicitly_accepted": false,
  "blockers": [],
  "passed": true
}


In [ ]:
from google.colab import runtime
runtime.unassign()

## 4. Phase 2 release

Phase 2 builds the immutable BM25 + E5 release from the accepted transcript artifacts.

Install the pinned retrieval stack below only after Phase 1 is finished. Because the notebook now runs everything in one Python process, **restart the Colab runtime once after this install**. All Phase 1 artifacts are already saved on Drive.

After the restart, rerun only the Drive/path cell in Section 1 and the **Import the corpus helpers** cell. Skip the Phase 1 `%pip` and NeMo runtime cells.

In [3]:
%pip install "transformers==4.52.0" "sentence-transformers==5.4.0" pytest

Reason for being yanked: <none given>
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 570.8/570.8 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 34.3 MB/s eta 0:00:00
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.6
    Uninstalling transformers-4.57.6:
      Successfully uninstalled transformers-4.57.6
  Attempting uninstall: sentence-transformers
    Found existing installation: sentence-transformers 5.7.0
    Uninstalling sentence-transformers-5.7.0:
      Successfully uninstalled sentence-transformers-5.7.0


### Build and validate

The build rechecks the Phase 1 acceptance boundary, loads E5 directly, calls the production `build_release()`, and validates the resulting hashes. The existing `CURRENT` release is reused for unchanged E5 rows when available, but `CURRENT` is not changed here.

In [5]:
print("transformers          :", importlib_metadata.version("transformers"))
print("sentence-transformers :", importlib_metadata.version("sentence-transformers"))

RELEASE_ID              = "aic2026-full-20260817-r01"
ALLOW_RESIDUAL_FAILURES = False
RELEASE_PATH            = RELEASES_ROOT / RELEASE_ID

if (RELEASE_PATH.exists()) :
    print("Release already exists; skipping immutable build:", RELEASE_PATH)
else :
    build_summary = build_corpus_release(
        artifact_root           = FULL_CORPUS_ROOT,
        release_id              = RELEASE_ID,
        allow_residual_failures = ALLOW_RESIDUAL_FAILURES,
    )
    print(json.dumps(build_summary, ensure_ascii = False, indent = 2))

transformers          : 4.52.0
sentence-transformers : 5.4.0
Loading E5 document encoder...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/128 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_xlm-roberta_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

Batches:   0%|          | 0/1633 [00:00<?, ?it/s]

Release     : aic2026-full-20260817-r01
Videos      : 1,478
Windows     : 26,163 physical / 26,117 searchable
Release path: /content/drive/MyDrive/aic26/asr_model_comparison/production_full_corpus/phase2/releases/aic2026-full-20260817-r01
{
  "created_at_utc": "2026-08-19T08:34:09.544198+00:00",
  "release_id": "aic2026-full-20260817-r01",
  "release_path": "/content/drive/MyDrive/aic26/asr_model_comparison/production_full_corpus/phase2/releases/aic2026-full-20260817-r01",
  "previous_release": null,
  "corpus_identity": "4109e8a68309ae33de9a70add93010008f0897ecf65a58af67f0193a1535ded1",
  "video_count": 1478,
  "physical_window_count": 26163,
  "eligible_window_count": 26117,
  "embedding_shape": [
    26117,
    1024
  ],
  "embedding_dtype": "float32",
  "transcripts_size_bytes": 209010777,
  "bm25_size_bytes": 29458490,
  "e5_size_bytes": 106975360,
  "release_size_bytes": 215341986,
  "phase1_acceptance": {
    "schema_version": "1.0",
    "accepted_at_utc": "2026-08-19T08:13:37.2

In [6]:
validation = validate_corpus_release(FULL_CORPUS_ROOT, RELEASE_ID)
print(json.dumps(validation, ensure_ascii = False, indent = 2))

build_summary_path = REPORTS_ROOT / "phase2_build_summary.json"
if (build_summary_path.is_file()) :
    build_summary = read_json(build_summary_path)
    storage_df = pd.DataFrame([
        {"artifact" : "Phase 1 transcripts", "GiB" : build_summary["transcripts_size_bytes"] / 1024**3},
        {"artifact" : "BM25", "GiB" : build_summary["bm25_size_bytes"] / 1024**3},
        {"artifact" : "E5 embeddings", "GiB" : build_summary["e5_size_bytes"] / 1024**3},
        {"artifact" : "Complete release", "GiB" : build_summary["release_size_bytes"] / 1024**3},
    ])
    display(storage_df)

{
  "validated_at_utc": "2026-08-19T08:34:17.290182+00:00",
  "release_id": "aic2026-full-20260817-r01",
  "release_path": "/content/drive/MyDrive/aic26/asr_model_comparison/production_full_corpus/phase2/releases/aic2026-full-20260817-r01",
  "corpus_identity": "4109e8a68309ae33de9a70add93010008f0897ecf65a58af67f0193a1535ded1",
  "video_count": 1478,
  "physical_window_count": 26163,
  "eligible_window_count": 26117,
  "embedding_shape": [
    26117,
    1024
  ],
  "embedding_dtype": "float32",
  "bm25_document_count": 26117,
  "verify_hashes": true,
  "passed": true
}


,artifact,GiB
0,Phase 1 transcripts,0.194656
1,BM25,0.027435
2,E5 embeddings,0.099629
3,Complete release,0.200553


## 5. Benchmark and activation

Load the normal production retrieval engine and run visible first-stage and reranked searches. The benchmark checks deterministic ordering, score/timing validity, supporting-window ownership, and the production candidate-pair cap.

If you later provide a CSV with `query` and `target_video_id`, the same cell can also calculate first-stage Recall@30/50/75/100. Activation stays disabled until you explicitly change the flag.

In [7]:
BENCHMARK_QUERIES = [
    "Đồng Nai",
    "năm con hổ con",
    "bốn phi hành gia mặc áo đen",
    "tàu vũ trụ nghiên cứu ánh sáng cực quang",
    "một giống hổ quý hiếm ở miền Nam",
]
BENCHMARK_REPETITIONS = 3
BENCHMARK_TOP_K        = 50
GROUND_TRUTH_FILE : Path | None = None

ground_truth = None
if (GROUND_TRUTH_FILE is not None) :
    ground_truth_df = pd.read_csv(GROUND_TRUTH_FILE)
    ground_truth    = ground_truth_df[["query", "target_video_id"]].to_dict("records")

benchmark_summary = benchmark_release(
    artifact_root = FULL_CORPUS_ROOT,
    release_id    = RELEASE_ID,
    queries       = BENCHMARK_QUERIES,
    repetitions   = BENCHMARK_REPETITIONS,
    top_k         = BENCHMARK_TOP_K,
    ground_truth  = ground_truth,
    device        = "cuda",
)

benchmark_df = pd.read_csv(REPORTS_ROOT / "benchmark_query_results.csv")
display(benchmark_df)
print(json.dumps(benchmark_summary["aggregate"], ensure_ascii = False, indent = 2))

if ("candidate_recall" in benchmark_summary) :
    print(json.dumps(benchmark_summary["candidate_recall"], ensure_ascii = False, indent = 2))

config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

Benchmark:   0%|          | 0/30 [00:00<?, ?search/s]

,query,mode,repetition,top_video_id,total_ms,bm25_ms,e5_encode_ms,dense_search_ms,first_stage_fusion_ms,video_aggregation_ms,candidate_selection_ms,bge_ms,candidate_fusion_ms,result_build_ms,candidate_pair_count,reranker_batch_size,validation_errors
0,Đồng Nai,first_stage,1,K13_V019,78.892665,1.806436,46.332700,22.515279,0.640865,3.952465,0.000000,0.000000,0.000000,3.421321,0,NaN,[]
1,Đồng Nai,first_stage,2,K13_V019,63.155149,0.566768,33.342070,23.699961,0.569263,1.642919,0.000000,0.000000,0.000000,3.116795,0,NaN,[]
2,Đồng Nai,first_stage,3,K13_V019,73.261733,0.606581,37.342343,28.885040,0.648393,1.959685,0.000000,0.000000,0.000000,3.580728,0,NaN,[]
3,năm con hổ con,first_stage,1,L21_V029,63.660241,0.749592,34.155348,23.234828,0.560217,1.522109,0.000000,0.000000,0.000000,3.216766,0,NaN,[]
4,năm con hổ con,first_stage,2,L21_V029,70.471873,0.620203,38.589247,25.558180,0.601575,1.690408,0.000000,0.000000,0.000000,3.197906,0,NaN,[]
5,năm con hổ con,first_stage,3,L21_V029,61.604134,0.645509,32.461450,22.847819,0.568867,1.566781,0.000000,0.000000,0.000000,3.294869,0,NaN,[]
6,bốn phi hành gia mặc áo đen,first_stage,1,K02_V026,66.848965,0.891852,37.617279,22.840825,0.541600,1.609398,0.000000,0.000000,0.000000,3.137323,0,NaN,[]
7,bốn phi hành gia mặc áo đen,first_stage,2,K02_V026,61.331073,0.891492,32.250698,22.733257,0.580391,1.533330,0.000000,0.000000,0.000000,3.115849,0,NaN,[]
8,bốn phi hành gia mặc áo đen,first_stage,3,K02_V026,66.327802,2.994890,35.330875,22.653514,0.498433,1.560066,0.000000,0.000000,0.000000,3.078950,0,NaN,[]
9,tàu vũ trụ nghiên cứu ánh sáng cực quang,first_stage,1,L21_V015,60.664480,0.716953,32.642704,21.843035,0.513251,1.552061,0.000000,0.000000,0.000000,3.180443,0,NaN,[]


{
  "first_stage": {
    "total_ms": {
      "mean": 66.9552144,
      "p50": 65.307199,
      "p95": 78.6123682,
      "max": 78.892665
    },
    "bm25_ms": {
      "mean": 1.193102,
      "p50": 0.755722,
      "p95": 3.0359542999999998,
      "max": 3.131771
    },
    "e5_encode_ms": {
      "mean": 35.6321544,
      "p50": 35.330875,
      "p95": 40.912282899999994,
      "max": 46.3327
    },
    "dense_search_ms": {
      "mean": 23.802223933333334,
      "p50": 22.847819,
      "p95": 27.605638099999997,
      "max": 28.88504
    },
    "first_stage_fusion_ms": {
      "mean": 0.5746326666666667,
      "p50": 0.569263,
      "p95": 0.651342,
      "max": 0.658223
    },
    "video_aggregation_ms": {
      "mean": 2.0627695333333333,
      "p50": 1.614979,
      "p95": 4.528755099999998,
      "max": 5.873432
    },
    "candidate_selection_ms": {
      "mean": 0.0,
      "p50": 0.0,
      "p95": 0.0,
      "max": 0.0
    },
    "bge_ms": {
      "mean": 0.0,
      "p50": 0.0,


In [8]:
ACTIVATE_RELEASE = False

if (ACTIVATE_RELEASE) :
    final_summary = activate_corpus_release(FULL_CORPUS_ROOT, RELEASE_ID)
    print(json.dumps(final_summary, ensure_ascii = False, indent = 2))
else :
    print("Activation is disabled. Review the release validation and benchmark results first.")

Activation is disabled. Review the release validation and benchmark results first.
